In [3]:
!pip install opencv-python
!pip install ultralytics

!yolo settings sync=False

  Using cached ultralytics-8.4.89-py3-none-any.whl.metadata (41 kB)
  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp39-cp39-win_amd64.whl.metadata (6.1 kB)
  Using cached polars-1.36.1-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.0.20-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.36.1-cp39-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.2.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
  Using cached contourpy-1.3.0-cp39-cp39-win_amd64.whl.metadata (5.4 kB)
  Using cached mpma

In [4]:
import cv2
import numpy as np
from ultralytics import YOLO
import csv
from pathlib import Path

# Загрузка модели и изображений
model = YOLO('D:/Data_env/Inf_cell_v26L-seg.pt')
input_dir = Path("D:/Data_env/cells")  # Папка с изображениями
output_csv = Path("D:/Data_env/cells/results.csv")  # Файл, куда запишется результат

# Расширения файлов изображений
image_extensions = ("*.jpg", "*.jpeg", "*.png")

# Создание таблицы результатов
with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["File Name",
                    "Total Pixels",
                    "Mask Pixels",
                    "Coverage Percentage",
                    "Objects Detected"])

# Поиск всех изображений для анализа
image_files = []
for ext in image_extensions:
    image_files.extend(input_dir.glob(ext))
print(f"Найдено изображений для обработки: {len(image_files)}")

# Основной цикл обработки
for img_path in image_files:
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Ошибка загрузки файла (пропущен): {img_path.name}")
            continue

        h, w, _ = img.shape
        total_pixels = h * w

        # Запуск сегментации для конкретного кадра
        results = model(img_path, verbose=False)

        # Создание пустой маски
        all_masks = np.zeros((h, w), dtype=np.uint8)
        object_count = 0

        for result in results:
            if result.masks is not None:
                object_count += len(result.masks.xy)
                # Отрисовка полигонов на маске
                for segments in result.masks.xy:
                    points = np.array(segments, dtype=np.int32)
                    cv2.fillPoly(all_masks, [points], 255)

        # Учет количества закрашенных пикселей маски
        mask_pixels = int(np.sum(all_masks == 255))

        # Вычисление покрытия
        coverage_pct = (
            (mask_pixels / total_pixels) * 100 if total_pixels > 0 else 0
        )

        # Заполнение результатов текущей обработки изображения в CSV
        with open(output_csv, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(
                [
                    img_path.name,
                    total_pixels,
                    mask_pixels,
                    round(coverage_pct, 4),
                    object_count,
                ]
            )

        print(f"✅ Обработан {img_path.name} | Покрытие: {coverage_pct:.2f}%")

    except Exception as e:
        print(f"Ошибка при обработке {img_path.name}: {e}")

print(f"Обработка завершена. Результаты сохранены в {output_csv.resolve()}")

Найдено изображений для обработки: 14
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_10.jpg | Покрытие: 53.94%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_4.jpg | Покрытие: 51.07%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_5.jpg | Покрытие: 64.34%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_6.jpg | Покрытие: 59.96%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_7.jpg | Покрытие: 69.61%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_8.jpg | Покрытие: 37.75%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_01_9.jpg | Покрытие: 62.14%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_10.jpg | Покрытие: 80.43%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_4.jpg | Покрытие: 70.02%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_5.jpg | Покрытие: 66.18%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_6.jpg | Покрытие: 63.84%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_7.jpg | Покрытие: 63.00%
✅ Обработан Glycyrrhiza+3120_4w_MB+AII_100x_04_8.jpg | Покрытие: 79.59%
✅ Обработан Glycyrrhiza+

---